In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import nltk
from nltk import FreqDist
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from collections import Counter
from itertools import chain
import contractions
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder
from nltk.collocations import TrigramAssocMeasures, TrigramCollocationFinder
from nltk import ngrams

In [ ]:
data = pd.read_csv("dataset/mb_data.csv")
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
_classes = data.type.unique()
print(_classes)

In [6]:
def show_class_distribution(
    data,
    x="type",
    figsize=(16, 4),
    title="Distribution of Personality Types",
    xticks_size=10,
    palette="husl",
):
    plt.figure(figsize=figsize)
    sns.countplot(x=x, data=data, palette=palette)
    plt.xlabel("Personality Types", size=15)
    plt.ylabel("Counts", size=15)
    plt.xticks(size=xticks_size)
    plt.title(title, size=20)
    plt.show()

In [ ]:
show_class_distribution(data, xticks_size=14)

In [8]:
def divide_types(df):
    df["E-I"] = ""
    df["N-S"] = ""
    df["F-T"] = ""
    df["J-P"] = ""
    for index, row in df.iterrows():
        row["E-I"] = "E" if row.type[0] == "E" else "I"
        row["N-S"] = "N" if row.type[1] == "N" else "S"
        row["F-T"] = "F" if row.type[2] == "F" else "T"
        row["J-P"] = "J" if row.type[3] == "J" else "P"
    return df


data = divide_types(data)

In [ ]:
show_class_distribution(
    data,
    x="E-I",
    title="Distribution of I & E",
    figsize=(9, 3),
    xticks_size=20,
    palette="icefire",
)

In [ ]:
show_class_distribution(
    data,
    x="N-S",
    title="Distribution of N & S",
    figsize=(9, 3),
    xticks_size=20,
    palette="cubehelix",
)

In [ ]:
show_class_distribution(
    data,
    x="F-T",
    title="Distribution of F & T",
    figsize=(9, 3),
    xticks_size=20,
    palette="viridis",
)

In [ ]:
show_class_distribution(
    data,
    x="J-P",
    title="Distribution of J & P",
    figsize=(9, 3),
    xticks_size=20,
    palette="flare",
)

In [ ]:
data.loc[7, "posts"]

#### Cleaning

In [14]:
def fix_contractions(df, column_name="posts", new_column="cleaned_post"):
    df[new_column] = df[column_name].apply(lambda x: contractions.fix(x))
    return df


data = fix_contractions(data)

In [15]:
def clean_data(df, column_name = "cleaned_post"):
    df[column_name] = df[column_name].apply(lambda x: x.lower())
    df[column_name] = df[column_name].apply(lambda x: re.sub(r'@([a-zA-Z0-9_]{1,50})', '', x))
    df[column_name] = df[column_name].apply(lambda x: re.sub(r'#([a-zA-Z0-9_]{1,50})', '', x))
    df[column_name] = df[column_name].apply(lambda x: re.sub(r'http[s]?://\S+', '', x))
    df[column_name] = df[column_name].apply(lambda x: re.sub(r'[^A-Za-z]+', ' ', x))
    df[column_name] = df[column_name].apply(lambda x: re.sub(r' +', ' ', x))
    df[column_name] = df[column_name].apply(lambda x: " ".join([word for word in x.split() if not len(word) <3]))
    return df

data = clean_data(data)

In [ ]:
data.loc[7, "cleaned_post"]

In [ ]:
data["words_count"] = data["cleaned_post"].apply(lambda x: len(x.split()))
data.head(5)

In [18]:
def plot_counts(df, column, xlabel):
    fig = plt.figure()
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    df[column].plot.hist(bins=25)

In [ ]:
plot_counts(data, column="words_count", xlabel="Words Count")

In [ ]:
data["char_count"] = data["cleaned_post"].apply(lambda x: len(x))
data.head(5)

In [ ]:
plot_counts(data, column="char_count", xlabel="Character Count")

In [23]:
stopword_list = stopwords.words("english")

In [24]:
def get_most_frequent(data, stop_words, column="cleaned_post", top=25):
    df = data[column].apply(
        lambda x: " ".join([word for word in x.split() if not word in stop_words])
    )
    counter = Counter(" ".join(df).split())
    return counter.most_common(top)

In [ ]:
most_frequents = get_most_frequent(data, stopword_list)
most_frequents[:10]

In [26]:
def show_most_frequents(most_frequent_words, top=20):
    most_frequent_df = pd.DataFrame(most_frequent_words)
    plt.figure(figsize=(16, 4))
    my_cmap = plt.get_cmap("viridis")
    plt.bar(
        x=most_frequent_df.iloc[:top, 0],
        height=most_frequent_df.iloc[:top, 1],
        color="slateblue",
    )
    plt.xlabel("Words", size=17)
    plt.ylabel("Counts", size=17)
    plt.title("Most Frequent Words", size=20)
    plt.show()

In [ ]:
show_most_frequents(most_frequents)

In [28]:
def show_wordcloud(data, stopword_list, column="cleaned_post"):
    fig = plt.figure(figsize=(15, 5))
    wordcloud = WordCloud(
        background_color="black", min_font_size=5, stopwords=stopword_list
    ).generate(data[column].to_string())
    plt.axis("off")
    plt.imshow(wordcloud)
    plt.show()

In [ ]:
show_wordcloud(data, stopword_list)

In [30]:
def show_sub_wordclouds(data, type_column, column, size, fig_size=(20, 15)):
    classes = data[type_column].unique()
    fig, ax = plt.subplots(len(classes), figsize=fig_size)
    j = 0
    for _class in classes:
        temp = data[data[type_column] == _class]
        wordcloud = WordCloud(background_color="black").generate(
            temp[column].to_string()
        )
        plt.subplot(*size, j + 1)
        plt.title(_class, size=25)
        plt.imshow(wordcloud)
        plt.axis("off")
        j += 1

In [ ]:
show_sub_wordclouds(data, type_column="type", column="cleaned_post", size=(4, 4))

In [ ]:
show_sub_wordclouds(
    data, type_column="E-I", column="cleaned_post", size=(1, 2), fig_size=(16, 8)
)

In [ ]:
show_sub_wordclouds(
    data, type_column="N-S", column="cleaned_post", size=(1, 2), fig_size=(16, 8)
)

In [ ]:
show_sub_wordclouds(
    data, type_column="F-T", column="cleaned_post", size=(1, 2), fig_size=(16, 8)
)

In [ ]:
show_sub_wordclouds(
    data, type_column="J-P", column="cleaned_post", size=(1, 2), fig_size=(16, 8)
)

In [36]:
def get_ngrams(data, n_gram, new_column, column="cleaned_post"):
    data["tokenized"] = data[column].apply(lambda x: x.split())
    data["sw_removal"] = data["tokenized"].apply(
        lambda x: [y for y in x if not y in stopword_list]
    )
    data[new_column] = data["sw_removal"].apply(lambda x: list(ngrams(x, n_gram)))
    data.drop(columns=["tokenized", "sw_removal"], inplace=True)
    return data

In [ ]:
data = get_ngrams(data, n_gram=2, new_column="bigrams")
data.head()

In [ ]:
data = get_ngrams(data, n_gram=3, new_column="trigrams")
data.head()

In [39]:
def most_common_ngram(data, column, top=20):
    temp = []
    for index, row in data.iterrows():
        temp += row[column]
    most_common = Counter(temp).most_common(top)
    return most_common

In [40]:
def plot_n_grams(ngrams, title, top=20):
    ngram_df = pd.DataFrame(ngrams)
    ngram_df.iloc[:, 0] = ngram_df.iloc[:, 0].astype(str)
    plt.figure(figsize=(7, 7))
    plt.barh(y=ngram_df.iloc[:top, 0], width=ngram_df.iloc[:top, 1])
    plt.xlabel("Counts", size=17)
    plt.ylabel("Pairs", size=17)
    plt.title(title, size=20)
    plt.show()

In [ ]:
bigrams_most_common = most_common_ngram(data, "bigrams")
bigrams_most_common

In [ ]:
plot_n_grams(bigrams_most_common, title="Most Frequent Bigrams")

In [ ]:
trigrams_most_common = most_common_ngram(data, "trigrams")
trigrams_most_common

In [ ]:
plot_n_grams(trigrams_most_common, title="Most Frequent Trigrams")

In [45]:
def remove_stopwords(data, stopword_list, column="cleaned_post"):
    data[column] = data[column].apply(word_tokenize)
    data[column] = data[column].apply(
        lambda x: [word for word in x if not word in stopword_list]
    )
    return data

In [46]:
def apply_lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(w) for w in text]

In [47]:
def lemmatize(data, stopword_list, column="cleaned_post"):
    data[column] = data[column].apply(apply_lemmatization)
    data[column] = data[column].apply(" ".join)
    return data

In [ ]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
data = remove_stopwords(data, stopword_list)

In [62]:
data = lemmatize(data, stopword_list)

In [ ]:
data.head()

In [ ]:
training_data = data[["cleaned_post", "E-I", "N-S", "F-T", "J-P"]].copy()
training_data.head(5)

In [68]:
def make_dummies(data, columns=["E-I", "N-S", "F-T", "J-P"]):
    for column in columns:
        temp_dummy = pd.get_dummies(data[column], prefix="type")
        data = data.join(temp_dummy)
    return data

In [ ]:
training_data = make_dummies(training_data)
training_data.head()